# 016 — Semantic Caching
**Production Infra** | Cut LLM costs with embedding-similarity cache

Traditional caches match exact strings.
**Semantic cache** matches queries that *mean the same thing*:

- "What is ML?" and "Define machine learning" → same cache hit
- Threshold: cosine similarity ≥ 0.92 → return cached answer

Saves 30-60% of LLM calls in typical Q&A applications.


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Build backend RAG pipeline ───────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
docs = [Document(page_content=c)
        for text in [ai_text, ml_text, dl_text]
        for c in splitter.split_text(text[:12000])]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using the context.\nContext:\n{ctx}\nQ: {q}\nA:"
)
def rag_answer(question: str) -> str:
    ctx = "\n\n".join(d.page_content for d in retriever.invoke(question))
    return (rag_prompt | llm | StrOutputParser()).invoke({"ctx": ctx[:3000], "q": question})

print(f"✓ RAG pipeline over {len(docs)} docs")


In [ ]:
# ── Semantic cache implementation ─────────────────────────────────────────
import numpy as np, time

class SemanticCache:
    def __init__(self, similarity_threshold=0.92):
        self.threshold = similarity_threshold
        self.cache: list[dict] = []   # [{query, vec, answer, hits}]
        self.total_calls  = 0
        self.cache_hits   = 0

    def _cosine(self, a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    def get(self, query: str) -> str | None:
        if not self.cache:
            return None
        q_vec  = np.array(embeddings.embed_query(query))
        best_sim, best_entry = 0.0, None
        for entry in self.cache:
            sim = self._cosine(q_vec, np.array(entry["vec"]))
            if sim > best_sim:
                best_sim, best_entry = sim, entry
        if best_sim >= self.threshold:
            best_entry["hits"] += 1
            return best_entry["answer"]
        return None

    def set(self, query: str, answer: str):
        q_vec = embeddings.embed_query(query)
        self.cache.append({"query": query, "vec": q_vec, "answer": answer, "hits": 0})

    def query(self, question: str, backend_fn) -> tuple[str, bool]:
        self.total_calls += 1
        cached = self.get(question)
        if cached is not None:
            self.cache_hits += 1
            return cached, True
        answer = backend_fn(question)
        self.set(question, answer)
        return answer, False

    @property
    def hit_rate(self):
        return self.cache_hits / self.total_calls if self.total_calls else 0

cache = SemanticCache(similarity_threshold=0.92)
print("✓ SemanticCache initialized (threshold=0.92)")


In [ ]:
# ── Warm up the cache ─────────────────────────────────────────────────────
seed_queries = [
    "What is machine learning?",
    "How does deep learning work?",
    "What are neural networks?",
    "What is supervised learning?",
]

print("Warming cache...")
for q in seed_queries:
    ans, hit = cache.query(q, rag_answer)
    print(f"  [{'HIT' if hit else 'MISS':4}] {q}")

print(f"\nCache size: {len(cache.cache)} entries")


In [ ]:
# ── Test with similar queries ──────────────────────────────────────────────
test_queries = [
    ("Define machine learning", True),      # should hit (similar to "What is ML?")
    ("Explain ML to a beginner", True),     # might hit
    ("What is reinforcement learning?", False),  # novel
    ("How does a neural network learn?", True),  # should hit
    ("Describe deep learning concepts", True),   # should hit
]

print("Testing semantic cache hits:\n")
for q, expected_hit in test_queries:
    t0  = time.time()
    ans, hit = cache.query(q, rag_answer)
    elapsed = time.time() - t0
    icon = "✓" if hit else "·"
    print(f"  {icon} [{'HIT' if hit else 'MISS':4}] {elapsed:.2f}s  {q}")
    print(f"      Answer: {ans[:120]}...")
    print()

print(f"Cache hit rate: {cache.hit_rate:.0%}  ({cache.cache_hits}/{cache.total_calls})")


In [ ]:
# ── Cost simulation ──────────────────────────────────────────────────────
import pandas as pd

# Rough token estimates
avg_input_tokens  = 800    # context + question
avg_output_tokens = 150    # answer
cost_per_1m_input  = 0.05  # USD for llama via Groq (approx)
cost_per_1m_output = 0.10

total_queries = 1000
hit_rate      = cache.hit_rate
llm_calls     = int(total_queries * (1 - hit_rate))

total_cost_no_cache = total_queries * (
    avg_input_tokens  / 1_000_000 * cost_per_1m_input +
    avg_output_tokens / 1_000_000 * cost_per_1m_output
)
total_cost_with_cache = llm_calls * (
    avg_input_tokens  / 1_000_000 * cost_per_1m_input +
    avg_output_tokens / 1_000_000 * cost_per_1m_output
)

print(f"Simulating {total_queries:,} queries at {cache.hit_rate:.0%} hit rate")
print(f"  Without cache : ${total_cost_no_cache:.4f}")
print(f"  With cache    : ${total_cost_with_cache:.4f}")
print(f"  Savings       : ${total_cost_no_cache - total_cost_with_cache:.4f} "
      f"({(1 - total_cost_with_cache/total_cost_no_cache):.0%})")


## Key Takeaways
- Semantic cache hit rate of **40-60%** is typical in enterprise Q&A bots
- Threshold tuning: lower (0.88) = more hits but risk stale answers; higher (0.95) = fewer hits, more accurate
- Production implementation: swap in-memory list for Redis with `HNSW` vector index
- Cache invalidation: TTL per entry + explicit purge when source docs change
